In [15]:
from ingest import load_faq_data
documents = load_faq_data()

In [16]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [17]:
documents = documents_llm

In [18]:
doc=documents[0]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [19]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [20]:
data_gen_instructions = """
You emulate a student who's taking our course.

Formulate 5 questions this student might ask based on a FAQ record. The record should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

Return a valid JSON object with this shape:
{"questions": ["question 1", "question 2", "question 3", "question 4", "question 5"]}

The output should resemble how people ask questions on the internet. Not too formal, not too short, not too long.
""".strip()

In [21]:
from dotenv import load_dotenv
import os
load_dotenv()
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [22]:
import json

user_prompt = json.dumps(doc)

In [23]:
messages = [
    {"role": "system", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    response_format={"type": "json_object"},
)

questions = Questions.model_validate_json(
    response.choices[0].message.content
)

In [11]:
response.usage

CompletionUsage(completion_tokens=133, prompt_tokens=249, total_tokens=382, completion_time=0.279612239, completion_tokens_details=None, prompt_time=0.015663251, prompt_tokens_details=None, queue_time=0.049282793, total_time=0.29527549)

In [12]:
questions.questions

["I'm behind schedule for the current project submissions, is it a strict deadline or is there some flexibility with it?",
 'If I miss out on submitting my project during the submission phase, are there any other ways to get a Certificate of Completion?',
 'Can I still interact with the instructors after the submission phase is over, maybe for feedback on my projects?',
 'Are there any additional projects or assignments after the initial ones, like an extra step or two in the learning process?',
 'Is there a minimum passing grade on the project submissions that I need to meet in order to get a certificate?']

In [24]:
from evaluation_utils import llm_structured

In [13]:
result = llm_structured(
    client,
    data_gen_instructions,
    user_prompt,
    model="llama-3.1-8b-instant"
)

In [15]:
result

Questions(questions=["Can you show examples of past projects from previous students so we know what's expected?", "What's the best way to get support if I'm stuck on a project outside of class hours?", "How do we submit our projects for grading if we're not in the same location?", 'Will our projects be reviewed for bugs or do we just need to show they work?', 'Is there a limit to how many times we can resubmit our projects before being disqualified from the course?'])

In [15]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': "What's the deadline to submit my project before the certificate isn't available?",
  'document': '74eb249bbf'},
 {'question': 'Will the course still provide a good experience even if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Is there a way to access course materials or projects without joining the course?',
  'document': '74eb249bbf'},
 {'question': 'Can late joiners interact with the rest of the class during course discussions?',
  'document': '74eb249bbf'},
 {'question': 'Will the course instructors provide support for those who are joining after the initial phase?',
  'document': '74eb249bbf'}]

In [25]:
import pandas as pd

In [17]:
pd.DataFrame(records)

,question,document
0,What's the deadline to submit my project befor...,74eb249bbf
1,Will the course still provide a good experienc...,74eb249bbf
2,Is there a way to access course materials or p...,74eb249bbf
3,Can late joiners interact with the rest of the...,74eb249bbf
4,Will the course instructors provide support fo...,74eb249bbf


In [26]:
from evaluation_utils import llm_structured_retry

In [27]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results

In [20]:
generate_ground_truth(doc)

[{'question': 'Is it too late to start the course, and will I still be able to submit a project in time to receive a certificate?',
  'document': '74eb249bbf'},
 {'question': "How do I make sure I'm eligible for a certificate in this course?",
  'document': '74eb249bbf'},
 {'question': "I've found the course, can I still join and submit a project for a certificate?",
  'document': '74eb249bbf'},
 {'question': 'What is the deadline for submitting a project to be eligible for a certificate in this course?',
  'document': '74eb249bbf'},
 {'question': 'Can I still join the course and receive a certificate as long as I submit my project on time?',
  'document': '74eb249bbf'}]

In [21]:
#running it for all documents (aka for each document, we generate 5 questions from that answer)
from tqdm.auto import tqdm

ground_truth = []

for doc in tqdm(documents[:5]):
    records = generate_ground_truth(doc)
    ground_truth.extend(records)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents[:10], generate_ground_truth)

#hit rate limit yikes

  0%|          | 0/10 [00:00<?, ?it/s]

In [33]:
ground_truth = []

for records in results:
    ground_truth.extend(records)

len(ground_truth)

50

In [34]:
df_ground_truth = pd.DataFrame(ground_truth)

In [36]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [64]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [65]:
df_ground_truth.head()

,question,course,document
0,Can I take this course at my own pace and stil...,llm-zoomcamp,69d122f12e
1,Is a certificate available if I complete the c...,llm-zoomcamp,69d122f12e
2,Do self-paced learners get any certificate for...,llm-zoomcamp,69d122f12e
3,Why are certificates not issued for the self-p...,llm-zoomcamp,69d122f12e
4,Is peer review of capstone projects required i...,llm-zoomcamp,69d122f12e


In [42]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [43]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [44]:
text_search("What is the difference between LLM and traditional machine learning?")

[{'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.',
  'doc_id': 'db78580409'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'Why does cosine similarity reduce to a matrix multiplication between the embeddings and the query vector?',
  'answer': 'Cosine similarity measures how aligned two vectors are, regardless of their magnitude. When all vectors (incl

In [45]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [46]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [50]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.',
  'doc_id': '0fab61eca2'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'What happens to code saved in Codesp

In [49]:
for d in results:
    print(f'{d["doc_id"]} == {doc_id}: {d["doc_id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [52]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [53]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [57]:
q = ground_truth[100]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Why does my OPENAI_API_KEY keep showing as missing inside Jupyter, and how do I fix it?


[1, 0, 0, 0, 0]

In [58]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [59]:
relevance_total_text = compute_relevance_total_text(ground_truth)

  0%|          | 0/395 [00:00<?, ?it/s]

In [60]:
relevance_total_text[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [62]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [63]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total[:15]

  0%|          | 0/395 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [66]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [67]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [68]:
cnt / len(example)
# 0.933

0.9333333333333333

In [69]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [71]:
hit_rate(relevance_total)
# 0.933

0.6506329113924051

In [72]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [73]:
total_score / len(example)
# 0.822

0.8222222222222222

In [74]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [75]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [76]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6708860759493671, 'mrr': 0.5849367088607595}

In [78]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [79]:
evaluate(
    ground_truth,
    text_search_v2
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6886075949367089, 'mrr': 0.6111814345991562}

In [77]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [80]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.7012658227848101, 'mrr': 0.6344303797468355}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.7113924050632912, 'mrr': 0.6418565400843883}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6708860759493671, 'mrr': 0.5849367088607595}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.6481012658227848, 'mrr': 0.5692405063291138}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.640506329113924, 'mrr': 0.5589451476793249}


In [81]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [82]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [83]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.749367,0.691392
19,2.0,4.0,0.2,0.749367,0.691392
35,5.0,10.0,0.5,0.749367,0.691392
34,5.0,10.0,0.2,0.746835,0.689536
33,5.0,10.0,0.1,0.746835,0.688903
4,1.0,2.0,0.2,0.744304,0.688565
18,2.0,4.0,0.1,0.746835,0.688143
20,2.0,4.0,0.5,0.744304,0.687932
6,1.0,4.0,0.1,0.746835,0.679283
7,1.0,4.0,0.2,0.749367,0.679241


In [84]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )